In [1]:
from typing import Dict, Any

In [2]:
def compile_production_prompt(user_variables: Dict[str, Any]) -> str:
    """
    Constructs a highly structured prompt template using distinct components:
    Role, Constraints, Context, and Few-Shot Patterns.
    """
    
    # 1. System Persona Definition
    role_block = (
        "ROLE:\n"
        "You are a strict, low-level Software Security Auditor specializing in "
        "reverse engineering and static binary analysis."
    )
    
    # 2. Hard Execution Constraints (Negative Prompting)
    constraints_block = (
        "CONSTRAINTS & RULES:\n"
        "- Analyze ONLY the assembly code provided in the text block.\n"
        "- Do NOT write introductory or concluding conversational prose (e.g., 'Sure, here is your analysis').\n"
        "- Output your final answer strictly in valid Markdown formatting.\n"
        "- If no security vulnerability is detected, write exactly: 'STATUS: SECURE'."
    )
    
    # 3. Few-Shot Examples (Teaches the model the exact structural layout we expect)
    few_shot_block = (
        "EXAMPLES OF EXPECTED OUTPUT:\n\n"
        "--- Example 1 ---\n"
        "INPUT ASSEMBLY:\n"
        "CMP EAX, 0x0\n"
        "JZ 0x00401123\n"
        "OUTPUT:\n"
        "STATUS: VULNERABLE\n"
        "- Type: Missing Null Check / Unconditional Jump Bypass\n"
        "- Description: The jump depends on EAX being zero, but code lacks validation prior to execution.\n"
        "--- End of Example 1 ---"
    )
    
    # 4. Dynamic Context Assembly (Where our live application variables are injected)
    context_block = f"""TARGET DATA TO ANALYZE:
The developer has submitted the following assembly instructions for review:
'''
{user_variables.get("assembly_code", "")}
'''"""

    # 5. Combining all components into a single structured string
    full_prompt = (
        f"{role_block}\n\n"
        f"{constraints_block}\n\n"
        f"{few_shot_block}\n\n"
        f"{context_block}\n\n"
        f"YOUR ANALYSIS:"
    )
    
    return full_prompt

In [3]:
# --- Simulating runtime execution ---

# Let's say a user feeds this assembly instruction segment into our application
raw_user_payload = {
    "assembly_code": (
        "MOV EAX, [ESP+4]\n"
        "CMP EAX, 0x100\n"
        "JG 0x00402245\n"
        "SUB EAX, 0x10"
    )
}

In [4]:
# Compile the final prompt string that will be sent to the LLM API
final_api_prompt = compile_production_prompt(raw_user_payload)

print("="*70)
print("COMPACTED PROMPT DELIVERED TO THE MODEL API:")
print("="*70)
print(final_api_prompt)

COMPACTED PROMPT DELIVERED TO THE MODEL API:
ROLE:
You are a strict, low-level Software Security Auditor specializing in reverse engineering and static binary analysis.

CONSTRAINTS & RULES:
- Analyze ONLY the assembly code provided in the text block.
- Do NOT write introductory or concluding conversational prose (e.g., 'Sure, here is your analysis').
- Output your final answer strictly in valid Markdown formatting.
- If no security vulnerability is detected, write exactly: 'STATUS: SECURE'.

EXAMPLES OF EXPECTED OUTPUT:

--- Example 1 ---
INPUT ASSEMBLY:
CMP EAX, 0x0
JZ 0x00401123
OUTPUT:
STATUS: VULNERABLE
- Type: Missing Null Check / Unconditional Jump Bypass
- Description: The jump depends on EAX being zero, but code lacks validation prior to execution.
--- End of Example 1 ---

TARGET DATA TO ANALYZE:
The developer has submitted the following assembly instructions for review:
'''
MOV EAX, [ESP+4]
CMP EAX, 0x100
JG 0x00402245
SUB EAX, 0x10
'''

YOUR ANALYSIS:
